# Continuous Batching

Wiki reference for [Continuous Batching](https://ml-viz-ruby.vercel.app/wiki/continuous-batching).

> **Tip:** use *File → Save a copy in Drive* so your edits persist.

**The idea in one sentence.** A static batch runs at the pace of its *longest* member — short
requests finish early and their GPU slots idle — while continuous (iteration-level) batching
re-decides batch membership **every decode step**, refilling freed slots from the queue
immediately. We implement both schedulers from scratch, reproduce the wiki's worked trace,
and show the one benchmarking gotcha that hides the entire effect.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(21)

## From scratch: two schedulers

Both simulators run in *iterations* (one model step each); every active sequence emits one token
per iteration. Requests are `(arrival_iter, output_len)` and admission is FIFO.

- **Static**: when the server is idle, take up to `slots` queued requests and run them until
  **all** finish; only then form the next batch.
- **Continuous**: every iteration, retire finished sequences and admit from the queue into any
  free slot before stepping.

We track each request's **first-token iteration** (its TTFT, in iterations), the **makespan**
(iteration when the last request finishes), and **slot utilization** (useful tokens ÷ slot-steps).

In [ ]:
def simulate(requests, slots, mode):
    """requests: list of (arrival, out_len), FIFO order. Returns metrics dict."""
    queue = sorted(range(len(requests)), key=lambda i: (requests[i][0], i))
    active = {}                          # req id -> tokens remaining
    first_token = {}
    t, done, slot_steps = 0, 0, 0
    while done < len(requests):
        t += 1
        can_admit = (not active) if mode == 'static' else (len(active) < slots)
        while can_admit and queue and requests[queue[0]][0] < t and len(active) < slots:
            i = queue.pop(0)
            active[i] = requests[i][1]
            first_token[i] = t           # first token emitted this iteration
        for i in list(active):           # one decode step for every active sequence
            active[i] -= 1
            if active[i] == 0:
                del active[i]
                done += 1
        slot_steps += slots
    useful = sum(r[1] for r in requests)
    ttft = np.array([first_token[i] - requests[i][0] for i in range(len(requests))])
    return {'makespan': t, 'utilization': useful / slot_steps, 'ttft': ttft}

demo = [(0, 3), (0, 8), (0, 2), (0, 5), (0, 4), (0, 3)]   # A..F from the wiki trace
for mode in ['static', 'continuous']:
    m = simulate(demo, slots=4, mode=mode)
    print(f"{mode:>10}: makespan {m['makespan']:2d} iters | "
          f"utilization {m['utilization']:5.1%} | mean TTFT {m['ttft'].mean():.1f}")

### Validate: the worked trace

Six requests (3, 8, 2, 5, 4, 3 output tokens), four slots, all arriving at once. The wiki's
timeline: C frees its slot after iteration 2 (E admitted at 3), A after iteration 3 (F admitted
at 4), and everything finishes when B does — **8 iterations**. Static batching runs
{A, B, C, D} for max(3,8,2,5) = 8 iterations, then {E, F} for 4 more: **12 iterations**.

In [ ]:
stat = simulate(demo, slots=4, mode='static')
cont = simulate(demo, slots=4, mode='continuous')

assert cont['makespan'] == 8,  'continuous finishes when B does (8 iters)'
assert stat['makespan'] == 12, 'static: 8-iteration batch then a 4-iteration batch'
assert cont['utilization'] > stat['utilization']
# E and F wait for the whole first batch under static (TTFT 9), but only for a slot under
# continuous (TTFT 3 and 4)
assert list(stat['ttft'][4:]) == [9, 9]
assert list(cont['ttft'][4:]) == [3, 4]
print('worked trace reproduced: 8 vs 12 iterations; E/F wait 3-4 iters instead of 9')

## Visualize: the win *is* the output-length variance

Same mean output length, growing spread — plus arrivals over time and more requests than slots.
With zero variance every batch member finishes together, so static and continuous are identical;
the gap opens exactly as fast as lengths diverge. This is also the benchmarking gotcha: synthetic
fixed-length load tests make the two schedulers look the same.

In [ ]:
mean_len, n_req, slots = 200, 64, 8
spreads = np.array([0, 25, 50, 100, 150, 190])
ratios = []
for sd in spreads:
    lens = np.clip(rng.normal(mean_len, sd, n_req), 10, None).astype(int)
    arrivals = np.sort(rng.integers(0, 400, n_req))
    reqs = list(zip(arrivals, lens))
    ratios.append(simulate(reqs, slots, 'static')['makespan']
                  / simulate(reqs, slots, 'continuous')['makespan'])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(spreads, ratios, 'o-', color='#4ade80', lw=2)
ax.axhline(1.0, color='#818cf8', ls='--', lw=1.5, label='no advantage')
ax.set(xlabel='std-dev of output length (mean = 200 tokens)',
       ylabel='static ÷ continuous makespan',
       title='Continuous batching wins exactly as much as lengths vary')
ax.legend()
plt.tight_layout(); plt.show()

assert ratios[0] < 1.15, 'near-zero variance: schedulers nearly tie (the benchmark gotcha)'
assert ratios[-1] > 1.3, 'high variance: continuous wins big'
assert ratios[-1] > ratios[0]
print('spread ->', dict(zip(spreads.tolist(), np.round(ratios, 2).tolist())))

**What to notice.**

- At spread 0 the ratio hugs 1.0 — **fixed-length benchmarks hide the entire benefit**. Real
  chat traffic has spreads comparable to the mean, out where the curve pays 1.5–2×+.
- The mechanism is visible in the utilization numbers from the trace: static idles slots from
  the moment the first batch member finishes; continuous keeps them ~full until the queue drains.
- TTFT improves even more than makespan: queued requests wait for *a slot* (one member
  finishing), not for *a whole batch*.
- Not modeled here: prefill cost. In a real engine, admitting a request injects its prompt's
  prefill into the step — un-chunked, a long prompt stalls every running request's ITL for that
  iteration, which is why **chunked prefill** spends the per-iteration token budget on decode
  first and prefill chunks with the remainder.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **fixed-length benchmarks** | static ≈ continuous (verified above) — always test with realistic length spread |
| **un-chunked prefills** | one 8k prompt admission spikes every co-batched request's ITL |
| **`max_num_seqs` as free throughput** | past the KV-read crossover, more concurrency mostly buys tail latency — sweep against goodput |
| **admission ignores KV headroom** | admitting on slot-count alone causes preemption storms; gate on free blocks too |
| **mean-latency dashboards** | prefill stalls and preemptions live entirely in the p95/p99 tail |

## ✏️ Your turn

**Exercise.** Implement `goodput(ttft_ms, itl_ms, ttft_slo, itl_slo)`: given per-request arrays
of TTFT and mean ITL (milliseconds), return the **fraction of requests meeting both SLOs**.
Goodput — not raw throughput — is the number that tells you when raising the batch size stops
helping: throughput keeps climbing while goodput turns over.

In [ ]:
def goodput(ttft_ms, itl_ms, ttft_slo, itl_slo):
    """
    ttft_ms, itl_ms: arrays, one entry per request
    returns: fraction of requests with ttft <= ttft_slo AND itl <= itl_slo
    """
    ttft_ms, itl_ms = np.asarray(ttft_ms), np.asarray(itl_ms)
    # TODO(you): boolean mask of requests meeting BOTH SLOs, then its mean
    return ...

print(goodput([100, 600, 300], [30, 30, 80], ttft_slo=500, itl_slo=50))

In [ ]:
# Assertion — passes silently when your implementation is correct
got = goodput([100, 600, 300], [30, 30, 80], 500, 50)
assert abs(got - 1 / 3) < 1e-9, f'only the first request meets both SLOs, got {got}'
assert goodput([1, 1], [1, 1], 10, 10) == 1.0
assert goodput([100, 100], [60, 60], 500, 50) == 0.0, 'ITL violations count too'
print('all checks passed — you are now measuring goodput, not vanity throughput')

<details><summary>Solution</summary>

```python
def goodput(ttft_ms, itl_ms, ttft_slo, itl_slo):
    ttft_ms, itl_ms = np.asarray(ttft_ms), np.asarray(itl_ms)
    ok = (ttft_ms <= ttft_slo) & (itl_ms <= itl_slo)
    return ok.mean()
```
</details>

## Key takeaways

- **Continuous batching = re-decide the batch every decode step**: retire, admit, preempt,
  step. No sequence ever waits for another sequence — only for resources.
- **The worked trace verifies the mechanism**: 8 vs 12 iterations, and queued requests' TTFT
  drops from 9 to 3–4 iterations because they wait for a slot, not a batch.
- **The win scales with output-length variance (verified)** — and vanishes on fixed-length
  synthetic benchmarks, so test with realistic traffic.
- **Chunked prefill** resolves the TTFT-vs-ITL tension inside the token budget; **goodput**
  (SLO-meeting throughput) is the metric that says when to stop raising concurrency.

**Next:** [the wiki page](https://ml-viz-ruby.vercel.app/wiki/continuous-batching) ·
[PagedAttention & KV-Cache Management](https://ml-viz-ruby.vercel.app/wiki/paged-attention) ·
[Optimizing LLM Inference](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/22-optimizing-llm-inference)